# Example 05: Speculative Decoding (Multi-Token Prediction)

Speeds up inference by predicting multiple tokens in parallel rather than one at a time.
This notebook includes a timing comparison so you can see the difference.

Requires a GPU runtime. Go to **Runtime**, then **Change runtime type**, and select **T4 GPU**.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import subprocess
subprocess.run([
    "curl", "-L",
    "https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/main/gemma-4-E2B-it.litertlm?download=true",
    "-o", "/content/gemma-4-E2B-it.litertlm"
], check=True)

In [ ]:
import litert_lm
import time

MODEL_PATH = "/content/gemma-4-E2B-it.litertlm"
PROMPT = "Write a short poem about the ocean."

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

# Run without speculative decoding first
print("Without speculative decoding:")
t0 = time.time()
with litert_lm.Engine(MODEL_PATH, backend=litert_lm.Backend.GPU) as engine:
    with engine.create_conversation() as conversation:
        response = conversation.send_message(PROMPT)
        print(response["content"][0]["text"])
print(f"Time: {time.time() - t0:.2f}s\n")

# Run with speculative decoding
print("With speculative decoding (MTP):")
t0 = time.time()
with litert_lm.Engine(
    MODEL_PATH,
    backend=litert_lm.Backend.GPU,
    enable_speculative_decoding=True,
) as engine:
    with engine.create_conversation() as conversation:
        response = conversation.send_message(PROMPT)
        print(response["content"][0]["text"])
print(f"Time: {time.time() - t0:.2f}s")